In [3]:
import sys
import numpy as np
import pandas as pd
import math
from sgp4.api import Satrec
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, CartesianRepresentation, CartesianDifferential
from astropy.utils.iers import conf
import astropy.units as u

conf.auto_download = False 
sys.path.append('../src')
from sensor import GSENSE4040
from geometry import calculate_tangent_point_ecef

H_SAT_APPROX = 500.0
MIN_TARGET_H = 20.0
FOCAL_LENGTH = 100.0

sensor = GSENSE4040(focal_length_mm=FOCAL_LENGTH)

s = '1 42017U 17008C   26114.50000000  .00000000  00000-0  50000-4 0  9998'
t = '2 42017  97.5000 100.0000 0010000   0.0000 180.0000 15.20000000500003'
satellite = Satrec.twoline2rv(s, t)

# 仿真时间范围 (生成 20 分钟的数据)
start_time = Time('2026-04-26T12:00:00', scale='utc')
time_steps = start_time + np.arange(0, 40) * 30 * u.second

results = []
print("正在执行动态姿态闭环跟瞄仿真...")

for t_current in time_steps:
    jd1, jd2 = t_current.jd1, t_current.jd2
    e, r_teme, v_teme = satellite.sgp4(jd1, jd2)
    if e != 0: continue 
        
    teme_rep = CartesianRepresentation(
        r_teme * u.km, 
        differentials=CartesianDifferential(v_teme * u.km / u.s)
    )
    teme_coord = TEME(teme_rep, obstime=t_current)
    itrs_coord = teme_coord.transform_to(ITRS(obstime=t_current))
    
    pos_ecef = [itrs_coord.x.value, itrs_coord.y.value, itrs_coord.z.value]
    vel_ecef = [
        itrs_coord.velocity.d_x.value, 
        itrs_coord.velocity.d_y.value, 
        itrs_coord.velocity.d_z.value
    ]
    
    # ==========================================
    # 核心升级：闭环动态姿态计算
    # ==========================================
    # 1. 传感器获取卫星此时此刻的极其精确的物理高度 (km)
    current_sat_alt = itrs_coord.earth_location.height.to_value(u.km)
    
    # 2. 飞控计算机实时计算：为了死死咬住 20km 边缘，现在脖子该往下倾斜多少度？
    dynamic_alpha = sensor.calc_center_angle_from_bottom_height(current_sat_alt, MIN_TARGET_H)
    
    # 3. 姿态控制系统执行该角度，获取实际切点数据
    tp_data = calculate_tangent_point_ecef(pos_ecef, vel_ecef, dynamic_alpha)
    
    results.append({
        'UTC_Time': t_current.isot,
        'Sat_Alt_km': round(current_sat_alt, 2),      # 观察卫星在轨道上如何“呼吸”起伏
        'Pitch_Angle': round(dynamic_alpha, 4),       # 观察相机的下俯角如何动态“点头”
        'Target_Lat': round(tp_data['lat'], 2),
        'Target_Alt_km': round(tp_data['h_tg'], 2),   # 观察切点高度是否被死死锁住
        'LOS_Dist_km': round(tp_data['distance'], 2)
    })

df_results = pd.DataFrame(results)
print("\n✅ 动态跟瞄运行完成！请观察 Pitch_Angle 和 Target_Alt_km 的变化：")
display(df_results.head(10))

正在执行动态姿态闭环跟瞄仿真...

✅ 动态跟瞄运行完成！请观察 Pitch_Angle 和 Target_Alt_km 的变化：


,UTC_Time,Sat_Alt_km,Pitch_Angle,Target_Lat,Target_Alt_km,LOS_Dist_km
0,2026-04-26T12:00:00.000,516.80,11.3336,-44.18,379.83,1353.07
1,2026-04-26T12:00:30.000,515.89,11.3149,-42.33,380.06,1350.82
2,2026-04-26T12:01:00.000,515.00,11.2963,-40.48,380.29,1348.59
3,2026-04-26T12:01:30.000,514.11,11.2779,-38.63,380.52,1346.37
4,2026-04-26T12:02:00.000,513.23,11.2596,-36.78,380.74,1344.18
5,2026-04-26T12:02:30.000,512.37,11.2417,-34.93,380.96,1342.03
6,2026-04-26T12:03:00.000,511.52,11.2241,-33.08,381.16,1339.91
7,2026-04-26T12:03:30.000,510.69,11.2068,-31.23,381.36,1337.84
8,2026-04-26T12:04:00.000,509.89,11.1901,-29.37,381.54,1335.83
9,2026-04-26T12:04:30.000,509.11,11.1738,-27.52,381.71,1333.87
